# 03c. Validar lista de ouro — mãe + endereço

Paralelo ao [`03b_avaliar_lista_ouro.ipynb`](03b_avaliar_lista_ouro.ipynb),
para o modelo v2.

Censo da [`LISTA_OURO_AVALIACAO_ARQUIVO`](../config.py) contra `cpf_limpo`
(CPFs da ouro **dentro**). JSON do
[`02c_treinar_splink_mae_endereco.ipynb`](02c_treinar_splink_mae_endereco.ipynb),
14 regras do [`02d_aplicar_splink_mae_endereco.ipynb`](02d_aplicar_splink_mae_endereco.ipynb),
`predict(0,25)` (piso do resgate baixo) e atribuição como no
[`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb)
(modelo ≥ 0,90; resgate R1/R2/R3; baixo R1/R2 sem R3; teto 3).

Métricas: mesmo pacote do [`README_validacao_ouro.md`](../README_validacao_ouro.md)
(`associou_cpf_ouro`, FP da associação/escolha, FN por corte). Predict grava
`splink_predictions_ouro_mae_endereco.parquet` (não sobrescreve o 03b).


In [1]:
import json
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from IPython.display import display

from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    LISTA_OURO_AVALIACAO_ARQUIVO,
    MAX_CENSOS_POR_CPF,
    OUTPUT_DIR,
    PREDICT_NUM_CHUNKS_LEFT,
    PREDICT_NUM_CHUNKS_RIGHT,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_MAE_ENDERECO,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    cpf_norm_sql,
    lista_ouro_colunas,
    drop_splink_temp_tables,
    ensure_output_dir,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO_MODELO = 0.90
PISO_RESGATE = 0.50
PISO_BAIXO = 0.25
TETO = MAX_CENSOS_POR_CPF
# Corte do predict da ouro = piso do resgate baixo.
T_PREDICT = PISO_BAIXO
# Alias usado nos gráficos Splink / prediction_errors (pacote clerical).
T = PISO_RESGATE
TOP_N = 20
PREDICTIONS_OURO = OUTPUT_DIR / 'splink_predictions_ouro_mae_endereco.parquet'
SPLINK_MODEL_JSON = SPLINK_MODEL_MAE_ENDERECO

print_paths()
print('LISTA_OURO_AVALIACAO_ARQUIVO:', LISTA_OURO_AVALIACAO_ARQUIVO)
print('PREDICTIONS_OURO:', PREDICTIONS_OURO)
print('SPLINK_MODEL_MAE_ENDERECO:', SPLINK_MODEL_MAE_ENDERECO)
require_input(LISTA_OURO_AVALIACAO_ARQUIVO, label='LISTA_OURO_AVALIACAO')
if not SPLINK_MODEL_MAE_ENDERECO.exists():
    raise RuntimeError(
        f'Modelo Splink não encontrado: {SPLINK_MODEL_MAE_ENDERECO}. '
        'Rode notebooks/02c_treinar_splink_mae_endereco.ipynb — este notebook não retreina.'
    )

con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA],
    notebook_origem='00b',
)
print(
    f'Faixas: modelo>={PISO_MODELO} | resgate>={PISO_RESGATE} | '
    f'baixo>={PISO_BAIXO} | teto={TETO} | predict>={T_PREDICT}'
)
print(
    'DuckDB: threads / memory_limit =',
    con.execute("SELECT current_setting('threads'), current_setting('memory_limit')").fetchone(),
    f'(defaults {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})',
)
print('Predict chunks L/R:', PREDICT_NUM_CHUNKS_LEFT, PREDICT_NUM_CHUNKS_RIGHT)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

## Lista de ouro

`DESCRIBE` do parquet. Colunas: `PERSON_ID_CENSO` e `CPF_NORM`.
G = pares 1:1 com os dois ids no limpo.


In [2]:
display(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_AVALIACAO_ARQUIVO}')").df())

ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_AVALIACAO_ARQUIVO}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)
con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{LISTA_OURO_AVALIACAO_ARQUIVO}')
WHERE {id_ouro} IS NOT NULL AND {cpf_ouro} IS NOT NULL
''')

display(con.execute('''
SELECT
    COUNT(*) AS n_pares,
    COUNT(DISTINCT id_censo) AS n_censo,
    COUNT(DISTINCT cpf_norm) AS n_cpf
FROM lista_ouro
''').df())


,column_name,column_type,null,key,default,extra
0,ID_MORADOR,VARCHAR,YES,None,None,None
1,end_cpf,VARCHAR,YES,None,None,None
2,cpf_cpf,VARCHAR,YES,None,None,None
3,end_responsavel,VARCHAR,YES,None,None,None
4,pes_nome,VARCHAR,YES,None,None,None
5,cpf_nome,VARCHAR,YES,None,None,None
6,pes_nasc,VARCHAR,YES,None,None,None
7,cpf_nasc,VARCHAR,YES,None,None,None


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_pares,n_censo,n_cpf
0,28172369,28172369,28088413


In [3]:
con.execute('''
CREATE OR REPLACE TABLE gt_lista AS
SELECT
    'censo_' || id_censo AS unique_id_censo,
    'cpf_' || cpf_norm AS unique_id_cpf,
    id_censo,
    cpf_norm
FROM lista_ouro
QUALIFY
    COUNT(*) OVER (PARTITION BY id_censo) = 1
    AND COUNT(*) OVER (PARTITION BY cpf_norm) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE gt_ouro AS
SELECT gt.*
FROM gt_lista gt
JOIN {TABELA_CENSO_LIMPA} c ON c.unique_id = gt.unique_id_censo
JOIN {TABELA_CPF_LIMPA} p ON p.unique_id = gt.unique_id_cpf
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_ouro AS
SELECT *
FROM {TABELA_CENSO_LIMPA}
WHERE unique_id IN (SELECT unique_id_censo FROM gt_ouro)
''')

n_lista = con.execute('SELECT COUNT(*) FROM lista_ouro').fetchone()[0]
n_1a1 = con.execute('SELECT COUNT(*) FROM gt_lista').fetchone()[0]
n_censo_limpo = con.execute(f'''
SELECT COUNT(*) FROM gt_lista g
JOIN {TABELA_CENSO_LIMPA} c ON c.unique_id = g.unique_id_censo
''').fetchone()[0]
n_cpf_limpo = con.execute(f'''
SELECT COUNT(*) FROM gt_lista g
JOIN {TABELA_CPF_LIMPA} p ON p.unique_id = g.unique_id_cpf
''').fetchone()[0]
n_ouro = con.execute('SELECT COUNT(*) FROM gt_ouro').fetchone()[0]
n_censo_ouro = con.execute('SELECT COUNT(*) FROM censo_ouro').fetchone()[0]

display(con.execute(f'''
SELECT
    {n_lista} AS n_lista,
    {n_1a1} AS n_1a1,
    {n_lista - n_1a1} AS n_nao_1a1,
    {n_censo_limpo} AS n_1a1_censo_no_limpo,
    {n_cpf_limpo} AS n_1a1_cpf_no_limpo,
    {n_ouro} AS n_ouro,
    {n_censo_ouro} AS n_censo_ouro
''').df())

if n_ouro == 0:
    raise RuntimeError(
        'Nenhum par 1:1 da lista caiu no limpo — confira o recorte do NB00.'
    )
print('G (n_ouro):', f'{n_ouro:,}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_lista,n_1a1,n_nao_1a1,n_1a1_censo_no_limpo,n_1a1_cpf_no_limpo,n_ouro,n_censo_ouro
0,28172369,28005791,166578,714805,727048,661722,661722


G (n_ouro): 661,722


Quadrantes no join Censo ouro × CPF ouro. Nulo não conta como igual.


In [4]:
display(con.execute(f'''
SELECT
    CASE
        WHEN ca.nome_completo_phon IS NOT NULL
         AND pb.nome_completo_phon IS NOT NULL
         AND ca.nome_completo_phon = pb.nome_completo_phon
        THEN 'nome_igual'
        ELSE 'nome_diferente'
    END AS nome,
    CASE
        WHEN ca.data_nascimento IS NOT NULL
         AND pb.data_nascimento IS NOT NULL
         AND ca.data_nascimento = pb.data_nascimento
        THEN 'dob_igual'
        ELSE 'dob_diferente'
    END AS dob,
    COUNT(*) AS n
FROM gt_ouro gt
JOIN {TABELA_CENSO_LIMPA} ca ON ca.unique_id = gt.unique_id_censo
JOIN {TABELA_CPF_LIMPA} pb ON pb.unique_id = gt.unique_id_cpf
GROUP BY 1, 2
ORDER BY 1, 2
''').df())


,nome,dob,n
0,nome_diferente,dob_igual,152038
1,nome_igual,dob_diferente,38509
2,nome_igual,dob_igual,471175


## Blocking e predict

Quatorze regras OR do 02d (iguais ao 02b + mãe no blocking). Predict com
`p ≥ 0,25` (piso do resgate baixo do 04 mãe+endereço).


In [5]:
from splink import block_on

blocking_rules = [
    block_on('nome_completo_phon'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'dia_nascimento', 'cod_municipio'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'ano_nascimento', 'cod_municipio'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento', 'sexo', 'cep'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento', 'sexo', 'cep'),
    (
        "l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.ano_nascimento = r.ano_nascimento "
        "AND l.mes_nascimento = r.mes_nascimento "
        "AND l.sexo = r.sexo "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) <= 2 "
        "AND least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon)) >= 6 "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) * 6 "
        "<= least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon))"
    ),
    (
        "l.primeiro_nome_phon = r.primeiro_nome_phon "
        "AND l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.logradouro_norm = r.logradouro_norm "
        "AND l.cep = r.cep "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.nome_mae_phon = r.nome_mae_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
]


In [6]:
from splink.blocking_analysis import count_comparisons_from_blocking_rules
from splink.internals.charts import CumulativeBlockingRuleComparisonsGeneratedChart

materialize_splink_input(
    con,
    censo_table='censo_ouro',
    cpf_table=TABELA_CPF_LIMPA,
)
db_api = get_splink_db_api(con)

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_chart = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_chart = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
print(f'Chart blocking: censo {n_censo_chart:,} | cpf {n_cpf_chart:,}')

censo_in = db_api.register('splink_censo')
censo_in.dataset_display_name = 'censo'
cpf_in = db_api.register('splink_cpf')
cpf_in.dataset_display_name = 'cpf'

# 1.0 conta todos os pares. O default do Splink 5 amostra 5%.
contagem = count_comparisons_from_blocking_rules(
    [censo_in, cpf_in],
    blocking_rules=blocking_rules,
    link_type='link_only',
    unique_id_column_name='unique_id',
    record_sample_proportion=1.0,
)
n_pares_blocking = contagem[-1]['cumulative_comparison_count']
print(f'Pares OR: {n_pares_blocking:,}')
CumulativeBlockingRuleComparisonsGeneratedChart(contagem)


splink_input → censo_ouro ∪ cpf_limpo
Chart blocking: censo 661,722 | cpf 8,447,170


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pares OR: 100,148,582


SplinkChart (use .to_html() or .save() to export HTML)

In [7]:
from splink import Linker

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_view = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_view = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
n_censo_tab = con.execute('SELECT COUNT(*) FROM censo_ouro').fetchone()[0]
n_cpf_tab = con.execute(f'SELECT COUNT(*) FROM {TABELA_CPF_LIMPA}').fetchone()[0]
assert n_censo_view == n_censo_tab and n_cpf_view == n_cpf_tab, (
    f'Linker fora do recorte ouro × cpf_limpo: '
    f'censo {n_censo_view} vs {n_censo_tab}; cpf {n_cpf_view} vs {n_cpf_tab}'
)
print('Linker: censo_ouro', f'{n_censo_view:,}', '| cpf_limpo', f'{n_cpf_view:,}')

with open(SPLINK_MODEL_JSON, encoding='utf-8') as file:
    data = json.load(file)
for comp in data['comparisons']:
    if comp.get('output_column_name') != 'nome_completo_phon':
        continue
    for nivel in comp['comparison_levels']:
        if nivel.get('tf_adjustment_column') == 'primeiro_nome_phon':
            nivel['tf_adjustment_column'] = 'nome_completo_phon'
            nivel.pop('disable_tf_exact_match_detection', None)
data['retain_intermediate_calculation_columns'] = False
data['retain_matching_columns'] = False
data['blocking_rules_to_generate_predictions'] = blocking_rules

censo_in = db_api.register('splink_censo')
censo_in.dataset_display_name = 'censo'
cpf_in = db_api.register('splink_cpf')
cpf_in.dataset_display_name = 'cpf'
linker = Linker([censo_in, cpf_in], data)


Linker: censo_ouro 661,722 | cpf_limpo 8,447,170


In [8]:
predict_kwargs = {'threshold_match_probability': T_PREDICT}
if PREDICT_NUM_CHUNKS_LEFT is not None:
    predict_kwargs['num_chunks_left'] = PREDICT_NUM_CHUNKS_LEFT
if PREDICT_NUM_CHUNKS_RIGHT is not None:
    predict_kwargs['num_chunks_right'] = PREDICT_NUM_CHUNKS_RIGHT

df_predict = linker.inference.predict(**predict_kwargs)
pred = df_predict.physical_name
print('predictions table:', pred)
print('pares:', con.execute(f'SELECT COUNT(*) FROM {pred}').fetchone()[0])

ensure_output_dir()
cols = [r[0] for r in con.execute(f'DESCRIBE {pred}').fetchall()]
keep = [
    c for c in ('unique_id_l', 'unique_id_r', 'match_probability', 'match_weight')
    if c in cols
]
con.execute(
    f"COPY (SELECT {', '.join(keep)} FROM {pred}) "
    f"TO '{PREDICTIONS_OURO}' (FORMAT PARQUET, COMPRESSION ZSTD)"
)
drop_splink_temp_tables(con)
print('Predictions ouro:', PREDICTIONS_OURO)
print('colunas:', keep)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Blocking time: 52.75 seconds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Predict time (post-blocking): 119.78 seconds


predictions table: __splink__df_predict_386714382
pares: 687619
Removidas 6 tabelas/views residuais __splink__*
Predictions ouro: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21/splink_predictions_ouro_mae_endereco.parquet
colunas: ['unique_id_l', 'unique_id_r', 'match_probability', 'match_weight']


## Atribuição

Mesma lógica do [`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb):
modelo ≥ 0,90; resgate com R1/R2/R3; baixo com R1/R2 (sem R3); teto 3 cumulativo.

`melhor_por_censo` = melhor elegível **antes** do teto (para FP da escolha).
`associacoes_unicas` = atribuição final após teto.


In [9]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{PREDICTIONS_OURO}')
''')

tok_ca = "string_split(ca.nome_completo_phon, ' ')"
tok_pb = "string_split(pb.nome_completo_phon, ' ')"
n_ca = f"len({tok_ca})"
n_pb = f"len({tok_pb})"
n_min = f"least({n_ca}, {n_pb})"
n_inter = f"len(list_intersect({tok_ca}, {tok_pb}))"
cobertura = f"({n_inter} * 1.0 / nullif({n_min}, 0))"
contido = (
    f"{n_min} >= 2 AND {n_ca} <> {n_pb} AND {n_inter} = {n_min}"
)
primeiro_ok = (
    "ca.primeiro_nome_phon IS NOT NULL AND pb.primeiro_nome_phon IS NOT NULL "
    "AND ("
    "ca.primeiro_nome_phon = pb.primeiro_nome_phon "
    "OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) <= 1"
    ")"
)
data_igual = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)
data_ambas = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento <> pb.data_nascimento"
)
delta_dias = (
    "abs(date_diff('day', "
    "CAST(ca.data_nascimento AS DATE), CAST(pb.data_nascimento AS DATE)))"
)
mesmo_dia_mes = (
    "day(CAST(ca.data_nascimento AS DATE)) = day(CAST(pb.data_nascimento AS DATE)) "
    "AND month(CAST(ca.data_nascimento AS DATE)) = month(CAST(pb.data_nascimento AS DATE))"
)
delta_ano = (
    "abs(year(CAST(ca.data_nascimento AS DATE)) "
    "- year(CAST(pb.data_nascimento AS DATE)))"
)
data_plausivel = (
    f"({delta_dias} <= 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 10)"
)
nome_muito_forte = (
    f"(ca.nome_completo_phon = pb.nome_completo_phon) OR ({contido})"
)
mae_presente = (
    "ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL"
)
jw_mae = "jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon)"

r1_medio = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.75))"
)
r1_baixo = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.80))"
)
r2 = (
    f"({nome_muito_forte}) AND ({data_ambas}) AND ({data_plausivel})"
)
r3_medio = (
    f"({primeiro_ok}) AND ({cobertura} >= 0.55) AND ({mae_presente}) "
    f"AND ({jw_mae} >= 0.75)"
)

desempate = (
    "("
    "CAST(ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL AS INTEGER) "
    "+ CAST("
    "ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL "
    "AND ca.logradouro_norm = pb.logradouro_norm AS INTEGER) "
    "+ CAST("
    "ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep AS INTEGER)"
    ")"
)

con.execute(f'''
CREATE OR REPLACE TABLE pares AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    {desempate} AS desempate,
    CASE
        WHEN ({r1_medio}) THEN 'R1'
        WHEN ({r2}) THEN 'R2'
        WHEN ({r3_medio}) THEN 'R3'
    END AS regra_medio,
    CASE
        WHEN ({r1_baixo}) THEN 'R1'
        WHEN ({r2}) THEN 'R2'
    END AS regra_baixo
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {PISO_BAIXO}
''')

# Melhor elegível antes do teto (FP da escolha).
con.execute(f'''
CREATE OR REPLACE TABLE melhor_por_censo AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM pares
WHERE match_probability >= {PISO_MODELO}
   OR (
        match_probability >= {PISO_RESGATE}
        AND match_probability < {PISO_MODELO}
        AND regra_medio IS NOT NULL
   )
   OR (
        match_probability >= {PISO_BAIXO}
        AND match_probability < {PISO_RESGATE}
        AND regra_baixo IS NOT NULL
   )
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, desempate DESC, unique_id_cpf
) = 1
''')

# Atribuição final (mesmo loop do 04).
_t = con.execute('''
SELECT table_type FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'atribuicao'
''').fetchone()
if _t:
    _k = 'VIEW' if _t[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_k} IF EXISTS atribuicao')

con.execute('''
CREATE OR REPLACE TABLE atribuicao (
    unique_id_censo VARCHAR,
    unique_id_cpf VARCHAR,
    match_probability DOUBLE,
    faixa VARCHAR,
    regra VARCHAR
)
''')

faixas = [
    ('modelo', PISO_MODELO, None, None),
    ('resgate', PISO_RESGATE, PISO_MODELO, 'regra_medio'),
    ('baixo', PISO_BAIXO, PISO_RESGATE, 'regra_baixo'),
]

for nome_faixa, piso, teto_nota, col_regra in faixas:
    if teto_nota is None:
        filtro_p = f'match_probability >= {piso}'
    else:
        filtro_p = (
            f'match_probability >= {piso} '
            f'AND match_probability < {teto_nota}'
        )
    if col_regra is None:
        filtro_regra = 'TRUE'
        regra_sql = "'score'"
    else:
        filtro_regra = f'{col_regra} IS NOT NULL'
        regra_sql = col_regra

    con.execute(f'''
    CREATE OR REPLACE TABLE faixa_base AS
    SELECT
        unique_id_censo,
        unique_id_cpf,
        match_probability,
        desempate,
        {regra_sql} AS regra
    FROM pares
    WHERE {filtro_p}
      AND {filtro_regra}
      AND unique_id_censo NOT IN (SELECT unique_id_censo FROM atribuicao)
      AND unique_id_cpf NOT IN (
          SELECT unique_id_cpf
          FROM atribuicao
          GROUP BY 1
          HAVING COUNT(*) >= {TETO}
      )
    ''')

    con.execute('''
    CREATE OR REPLACE TABLE faixa_melhor AS
    SELECT unique_id_censo, unique_id_cpf, match_probability, regra
    FROM faixa_base
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY unique_id_censo
        ORDER BY match_probability DESC, desempate DESC, unique_id_cpf
    ) = 1
    ''')

    antes = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
    con.execute(f'''
    INSERT INTO atribuicao
    SELECT
        m.unique_id_censo,
        m.unique_id_cpf,
        m.match_probability,
        '{nome_faixa}' AS faixa,
        m.regra
    FROM faixa_melhor m
    JOIN (
        SELECT unique_id_cpf, COUNT(*) AS n_novo
        FROM faixa_melhor
        GROUP BY 1
    ) g ON g.unique_id_cpf = m.unique_id_cpf
    LEFT JOIN (
        SELECT unique_id_cpf, COUNT(*) AS n_ja
        FROM atribuicao
        GROUP BY 1
    ) j ON j.unique_id_cpf = m.unique_id_cpf
    WHERE coalesce(j.n_ja, 0) + g.n_novo <= {TETO}
    ''')
    depois = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
    print(f'{nome_faixa} ({filtro_p}): +{depois - antes:,} (total {depois:,})')

con.execute('''
CREATE OR REPLACE TABLE associacoes_unicas AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM atribuicao
''')

n_pred = con.execute(
    'SELECT COUNT(DISTINCT unique_id_censo) FROM splink_predictions'
).fetchone()[0]
n_melhor = con.execute('SELECT COUNT(*) FROM melhor_por_censo').fetchone()[0]
n_unicas = con.execute('SELECT COUNT(*) FROM associacoes_unicas').fetchone()[0]
print('Censos com par no predict:', f'{n_pred:,}')
print('Censos elegíveis (antes do teto):', f'{n_melhor:,}')
print('Associações (após teto):', f'{n_unicas:,}')

display(con.execute('''
SELECT faixa, regra, n
FROM (
    SELECT faixa, regra, COUNT(*) AS n
    FROM atribuicao
    GROUP BY 1, 2
)
ORDER BY
    CASE faixa
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
    END,
    regra
''').df())


modelo (match_probability >= 0.9): +590,519 (total 590,519)
resgate (match_probability >= 0.5 AND match_probability < 0.9): +9,315 (total 599,834)
baixo (match_probability >= 0.25 AND match_probability < 0.5): +10,576 (total 610,410)
Censos com par no predict: 624,883
Censos elegíveis (antes do teto): 610,410
Associações (após teto): 610,410


,faixa,regra,n
0,modelo,score,590519
1,resgate,R1,7654
2,resgate,R2,1507
3,resgate,R3,154
4,baixo,R1,9990
5,baixo,R2,586


## Pacote de métricas

Cada Censo de G em um corte. FN = não `associou_cpf_ouro`. FP da associação =
errado entre quem associou. FP da escolha = `M ≠ Y` entre quem tem `melhor_por_censo`.


In [10]:
con.execute('''
CREATE OR REPLACE TABLE avaliacao_ouro AS
SELECT
    gt.unique_id_censo,
    gt.unique_id_cpf AS unique_id_cpf_ouro,
    a.unique_id_cpf AS unique_id_cpf_associado,
    m.unique_id_cpf AS unique_id_cpf_escolha,
    a.match_probability AS p_associado,
    m.match_probability AS p_escolha,
    CASE
        WHEN a.unique_id_cpf = gt.unique_id_cpf THEN 'associou_cpf_ouro'
        WHEN a.unique_id_censo IS NOT NULL THEN 'associou_cpf_errado'
        WHEN m.unique_id_censo IS NOT NULL THEN 'melhor_par_mas_cpf_repartilhado'
        WHEN p.unique_id_censo IS NOT NULL THEN 'teve_par_abaixo_de_T'
        ELSE 'nenhum_par'
    END AS corte
FROM gt_ouro gt
LEFT JOIN associacoes_unicas a ON a.unique_id_censo = gt.unique_id_censo
LEFT JOIN melhor_por_censo m ON m.unique_id_censo = gt.unique_id_censo
LEFT JOIN (
    SELECT DISTINCT unique_id_censo FROM splink_predictions
) p ON p.unique_id_censo = gt.unique_id_censo
''')

display(con.execute('''
SELECT corte, COUNT(*) AS n
FROM avaliacao_ouro
GROUP BY 1
ORDER BY 1
''').df())


,corte,n
0,associou_cpf_errado,3825
1,associou_cpf_ouro,606585
2,nenhum_par,36839
3,teve_par_abaixo_de_T,14473


In [11]:
display(con.execute('''
SELECT
    COUNT(*) AS n_ouro,
    COUNT(*) FILTER (WHERE corte = 'associou_cpf_ouro') AS n_associou_cpf_ouro,
    COUNT(*) FILTER (WHERE corte = 'associou_cpf_errado') AS n_associou_cpf_errado,
    COUNT(*) FILTER (WHERE corte = 'melhor_par_mas_cpf_repartilhado')
        AS n_melhor_par_mas_cpf_repartilhado,
    COUNT(*) FILTER (WHERE corte = 'teve_par_abaixo_de_T') AS n_teve_par_abaixo_de_T,
    COUNT(*) FILTER (WHERE corte = 'nenhum_par') AS n_nenhum_par,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE corte = 'associou_cpf_ouro')
        / NULLIF(COUNT(*), 0),
        2
    ) AS recall_pct,
    ROUND(
        100.0 * (
            COUNT(*) - COUNT(*) FILTER (WHERE corte = 'associou_cpf_ouro')
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS fn_pct,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE corte = 'associou_cpf_ouro')
        / NULLIF(
            COUNT(*) FILTER (WHERE corte IN ('associou_cpf_ouro', 'associou_cpf_errado')),
            0
        ),
        2
    ) AS precisao_associacao_pct,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE corte = 'associou_cpf_errado')
        / NULLIF(
            COUNT(*) FILTER (WHERE corte IN ('associou_cpf_ouro', 'associou_cpf_errado')),
            0
        ),
        2
    ) AS fp_associacao_pct
FROM avaliacao_ouro
''').df())


,n_ouro,n_associou_cpf_ouro,n_associou_cpf_errado,n_melhor_par_mas_cpf_repartilhado,n_teve_par_abaixo_de_T,n_nenhum_par,recall_pct,fn_pct,precisao_associacao_pct,fp_associacao_pct
0,661722,606585,3825,0,14473,36839,91.67,8.33,99.37,0.63


In [12]:
display(con.execute('''
SELECT
    COUNT(*) AS n_com_escolha,
    COUNT(*) FILTER (WHERE unique_id_cpf_escolha <> unique_id_cpf_ouro)
        AS n_escolha_errada,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE unique_id_cpf_escolha <> unique_id_cpf_ouro)
        / NULLIF(COUNT(*), 0),
        2
    ) AS fp_escolha_pct
FROM avaliacao_ouro
WHERE unique_id_cpf_escolha IS NOT NULL
''').df())

display(con.execute('''
SELECT
    COUNT(*) AS n_ouro,
    COUNT(*) FILTER (WHERE p.unique_id_censo IS NOT NULL) AS n_par_ouro_no_predict,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE p.unique_id_censo IS NOT NULL)
        / NULLIF(COUNT(*), 0),
        2
    ) AS recall_par_ouro_predict_pct
FROM gt_ouro gt
LEFT JOIN splink_predictions p
    ON p.unique_id_censo = gt.unique_id_censo
   AND p.unique_id_cpf = gt.unique_id_cpf
''').df())


,n_com_escolha,n_escolha_errada,fp_escolha_pct
0,610410,3825,0.63


,n_ouro,n_par_ouro_no_predict,recall_par_ouro_predict_pct
0,661722,623206,94.18


Mesmo pacote nos quadrantes nome × DOB.


In [13]:
display(con.execute(f'''
SELECT
    CASE
        WHEN ca.nome_completo_phon IS NOT NULL
         AND pb.nome_completo_phon IS NOT NULL
         AND ca.nome_completo_phon = pb.nome_completo_phon
        THEN 'nome_igual'
        ELSE 'nome_diferente'
    END AS nome,
    CASE
        WHEN ca.data_nascimento IS NOT NULL
         AND pb.data_nascimento IS NOT NULL
         AND ca.data_nascimento = pb.data_nascimento
        THEN 'dob_igual'
        ELSE 'dob_diferente'
    END AS dob,
    COUNT(*) AS n_ouro,
    COUNT(*) FILTER (WHERE av.corte = 'associou_cpf_ouro') AS n_associou_cpf_ouro,
    COUNT(*) FILTER (WHERE av.corte = 'associou_cpf_errado') AS n_associou_cpf_errado,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE av.corte = 'associou_cpf_ouro')
        / NULLIF(COUNT(*), 0),
        2
    ) AS recall_pct,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE av.corte = 'associou_cpf_errado')
        / NULLIF(
            COUNT(*) FILTER (
                WHERE av.corte IN ('associou_cpf_ouro', 'associou_cpf_errado')
            ),
            0
        ),
        2
    ) AS fp_associacao_pct
FROM avaliacao_ouro av
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = av.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = av.unique_id_cpf_ouro
GROUP BY 1, 2
ORDER BY 1, 2
''').df())


,nome,dob,n_ouro,n_associou_cpf_ouro,n_associou_cpf_errado,recall_pct,fp_associacao_pct
0,nome_diferente,dob_igual,152038,111653,1263,73.44,1.12
1,nome_igual,dob_diferente,38509,25814,508,67.03,1.93
2,nome_igual,dob_igual,471175,469118,2054,99.56,0.44


## Gráficos Splink (`clerical_match_score`)

Ranking **pairwise**, não a associação 1:1 do 04. Precisão daqui ≠ `associou_cpf_ouro / (ouro + errado)`.

Rótulo: par ouro = 1; outro CPF do mesmo Censo no predict = 0. Sem negativo a ROC não existe. Segundo Linker só pontua esses pares (`retain_*` ligado).

FP/FN do `prediction_errors` são do par (outro candidato com `p ≥ T`), não de `associou_cpf_errado`.


In [14]:
con.execute('''
CREATE OR REPLACE TABLE labels_clericais AS
SELECT
    'censo' AS source_dataset_l,
    unique_id_censo AS unique_id_l,
    'cpf' AS source_dataset_r,
    unique_id_cpf AS unique_id_r,
    1.0 AS clerical_match_score
FROM gt_ouro
UNION ALL
SELECT
    'censo',
    p.unique_id_censo,
    'cpf',
    p.unique_id_cpf,
    0.0
FROM splink_predictions p
JOIN gt_ouro gt ON gt.unique_id_censo = p.unique_id_censo
WHERE p.unique_id_cpf <> gt.unique_id_cpf
''')

display(con.execute('''
SELECT
    clerical_match_score,
    COUNT(*) AS n
FROM labels_clericais
GROUP BY 1
ORDER BY 1 DESC
''').df())


,clerical_match_score,n
0,1.0,661722
1,0.0,64413


In [15]:
from splink import Linker

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')

with open(SPLINK_MODEL_JSON, encoding='utf-8') as file:
    data_cl = json.load(file)
for comp in data_cl['comparisons']:
    if comp.get('output_column_name') != 'nome_completo_phon':
        continue
    for nivel in comp['comparison_levels']:
        if nivel.get('tf_adjustment_column') == 'primeiro_nome_phon':
            nivel['tf_adjustment_column'] = 'nome_completo_phon'
            nivel.pop('disable_tf_exact_match_detection', None)
data_cl['retain_intermediate_calculation_columns'] = True
data_cl['retain_matching_columns'] = True
data_cl['blocking_rules_to_generate_predictions'] = blocking_rules

drop_splink_temp_tables(con)
db_api = get_splink_db_api(con)
censo_in = db_api.register('splink_censo')
censo_in.dataset_display_name = 'censo'
cpf_in = db_api.register('splink_cpf')
cpf_in.dataset_display_name = 'cpf'
linker_cl = Linker([censo_in, cpf_in], data_cl)
print('Linker clerical: censo_ouro × cpf_limpo (retain ligado; não faz predict cheio)')


Linker clerical: censo_ouro × cpf_limpo (retain ligado; não faz predict cheio)


In [18]:
from splink.internals.charts import (
    PrecisionRecallChart,
    ROCChart,
    ThresholdSelectionToolChart,
)

tab = linker_cl.evaluation.accuracy_analysis_from_labels_table(
    'labels_clericais',
    threshold_match_probability=0.5,
    match_weight_round_to_nearest=0.1,
    output_type='table',
    add_metrics=['f1'],
)
recs = tab.as_record_list()
if not recs:
    raise RuntimeError(
        'Truth table vazia. Confira source_dataset em labels_clericais '
        '(esperado censo / cpf) contra o concat do Linker.'
    )

recs_tool = [d for d in recs if d['precision'] > 0.5 and d['recall'] > 0.5]
if recs_tool:
    display(ThresholdSelectionToolChart(recs, add_metrics=['f1']))
else:
    print('Threshold tool sem pontos com precisão e recall > 0,5; ROC e PR abaixo.')
display(ROCChart(recs))
display(PrecisionRecallChart(recs))

df_truth = tab.as_pandas_dataframe()
i = (df_truth['match_probability'] - T).abs().idxmin()
print('Linha mais perto de T =', T, '(precisão/recall pairwise, não da associação)')
display(
    df_truth.loc[
        [i],
        [
            'truth_threshold',
            'match_probability',
            'tp',
            'fp',
            'fn',
            'tn',
            'precision',
            'recall',
            'f1',
        ],
    ]
)


SplinkChart (use .to_html() or .save() to export HTML)

SplinkChart (use .to_html() or .save() to export HTML)

SplinkChart (use .to_html() or .save() to export HTML)

Linha mais perto de T = 0.5 (precisão/recall pairwise, não da associação)


,truth_threshold,match_probability,tp,fp,fn,tn,precision,recall,f1
179,0.0,0.5,605371.0,37118.0,56351.0,27295.0,0.942228,0.914842,0.928333


In [26]:
erros = linker_cl.evaluation.prediction_errors_from_labels_table(
    'labels_clericais',
    include_false_positives=True,
    include_false_negatives=True,
    threshold_match_probability=0.90,
)
tab_erros = erros.physical_name
print('prediction_errors:', tab_erros, '| corte T =', T)
display(con.execute(f'''
SELECT truth_status, COUNT(*) AS n
FROM {tab_erros}
GROUP BY 1
ORDER BY 1
''').df())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

prediction_errors: __splink__labels_with_fp_fn_status_c8a14a0f7 | corte T = 0.5


,truth_status,n
0,FN,70817
1,FP,12780


In [20]:
fn = con.execute(f'''
SELECT *
FROM {tab_erros}
WHERE truth_status = 'FN'
ORDER BY match_probability
LIMIT {TOP_N}
''').df()
display(fn[[
    'unique_id_l', 'unique_id_r', 'match_probability',
    'clerical_match_score', 'truth_status',
]])
if fn.empty:
    print('Nenhum FN pairwise em T.')
else:
    display(linker_cl.visualisations.waterfall_chart(fn.to_dict(orient='records')))


,unique_id_l,unique_id_r,match_probability,clerical_match_score,truth_status
0,censo_2111300050011390170060000180000002000000003,cpf_61363902300,4.866174e-07,1.0,FN
1,censo_2111904050000170000090000118000002000000003,cpf_05343059112,4.866174e-07,1.0,FN
2,censo_2105302050001590030040000449000001000000003,cpf_07645386320,4.888344e-07,1.0,FN
3,censo_2108603050000010100020000101000002000000002,cpf_62564118380,5.025721e-07,1.0,FN
4,censo_2111300050012650020050000007000001000000002,cpf_60707963303,5.085267e-07,1.0,FN
5,censo_2111300050015170160030000180000001000000004,cpf_61243331399,5.085267e-07,1.0,FN
6,censo_2111300050006760160040000328000001000000003,cpf_61563219344,5.109483e-07,1.0,FN
7,censo_2107506050003230020010000063000001000000002,cpf_06145047321,5.121677e-07,1.0,FN
8,censo_2111300050017250040060000136000001000000002,cpf_00835690300,5.121677e-07,1.0,FN
9,censo_2111300050000170070040000344000001000000004,cpf_62355518351,5.121677e-07,1.0,FN


SplinkChart (use .to_html() or .save() to export HTML)

In [21]:
fp = con.execute(f'''
SELECT *
FROM {tab_erros}
WHERE truth_status = 'FP'
ORDER BY match_probability DESC
LIMIT {TOP_N}
''').df()
print('FP pairwise (não é associou_cpf_errado)')
display(con.execute(f'''
SELECT
    e.unique_id_l,
    e.unique_id_r,
    gt.unique_id_cpf AS unique_id_cpf_ouro,
    e.match_probability,
    e.clerical_match_score,
    e.truth_status,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ouro.nome_completo AS nome_cpf_ouro,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ouro.data_nascimento AS dob_cpf_ouro
FROM {tab_erros} e
JOIN gt_ouro gt ON gt.unique_id_censo = e.unique_id_l
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = e.unique_id_l
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = e.unique_id_r
JOIN {SPLINK_INPUT_VIEW} ouro ON ouro.unique_id = gt.unique_id_cpf
WHERE e.truth_status = 'FP'
ORDER BY e.match_probability DESC, e.unique_id_l
LIMIT {TOP_N}
''').df())
if fp.empty:
    print('Nenhum FP pairwise em T.')
else:
    display(linker_cl.visualisations.waterfall_chart(fp.to_dict(orient='records')))


FP pairwise (não é associou_cpf_errado)


,unique_id_l,unique_id_r,unique_id_cpf_ouro,match_probability,clerical_match_score,truth_status,nome_censo,nome_cpf,nome_cpf_ouro,dob_censo,dob_cpf,dob_cpf_ouro
0,censo_2110856050000160110010000138000001000000007,cpf_06041422347,cpf_61090744382,1.0,0.0,FP,ISADORA KAROLINE SOUSA MELO,ISADORA KAROLINE SOUSA MELO,ISADORA KAROLINE SOUSA MELO,2002-03-09,2002-03-09,2002-03-09
1,censo_2105427050000090090020000276000001000000003,cpf_03733565363,cpf_61053697350,1.0,0.0,FP,ADRIELY RODRIGUES REIS,ADRIELLY RODRIGUES REIS,ADRIELY RODRIGUES REIS,1996-05-17,1996-05-17,1996-05-17
2,censo_2111201050001320030020000020000001000000002,cpf_61009659332,cpf_60503859397,1.0,0.0,FP,JARDILENA BARBOSA SANTOS,JARDILENA BARBOSA SANTOS,JARDILENA BARBOSA SANTOS,1995-06-26,1995-06-26,1995-06-26
3,censo_2105302050004280070010000185000001000000002,cpf_34325018387,cpf_43618677391,1.0,0.0,FP,IVALDO VIEIRA SOUSA,IVALDO VIEIRA SOUSA,IVALDO VIEIRA SOUSA,1968-08-11,1968-08-11,1968-08-11
4,censo_2108058050000380210020000104000001000000003,cpf_61186614307,cpf_05581318328,1.0,0.0,FP,ANTONIO JOSE SOUSA CONCEICAO,ANTONIO JOSE SOUSA CONCEICAO,ANTONIO JOSE SOUSA CONCEICAO,1995-08-20,1995-08-20,1961-12-25
5,censo_2113009050000060040020000238000001000000002,cpf_62195335343,cpf_60439132320,1.0,0.0,FP,JONAS SILVA CARVALHO,JONAS SILVA CARVALHO,JONAS SILVA CARVALHO,1998-08-08,1998-08-08,1998-08-08
6,censo_2106326050000030120030000274000001000000003,cpf_05977462301,cpf_05217671300,1.0,0.0,FP,EUZIANE CASTRO SANTOS,EUSIENE CASTRO SANTOS,EUSIANE CASTRO SANTOS,1991-07-05,1991-07-05,1991-07-05
7,censo_2112506050000740140010000180000001000000003,cpf_61971746347,cpf_61971748390,1.0,0.0,FP,LAINE OLIVEIRA REIS,LANE OLIVEIRA REIS,LAINE OLIVEIRA REIS,1998-04-13,1998-04-13,1998-04-13
8,censo_2101608050002640010010000019000001000000002,cpf_04133128362,cpf_04220627316,1.0,0.0,FP,WISLLANDYA QUIRINO ROCHA,ISLANDIA QUIRINO ROCHA,WISLLANDYA QUIRINO ROCHA,2001-03-20,2001-03-20,2001-03-20
9,censo_2101509050000260270010000197000001000000003,cpf_08427594330,cpf_61779168306,1.0,0.0,FP,DAMIANA CARVALHO SILVA,DAMIANA CARVALHO SILVA,DAMIAO CARVALHO SILVA,1998-04-11,1998-04-11,1998-04-11


SplinkChart (use .to_html() or .save() to export HTML)

## Pares associados ao CPF errado

`A ≠ Y`: a atribuição publicou **outro** CPF que o da ouro.


In [22]:
cols_exemplo = '''
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ouro.nome_completo AS nome_cpf_ouro,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ouro.data_nascimento AS dob_cpf_ouro,
    ca.nome_mae AS mae_censo,
    pb.nome_mae AS mae_cpf,
    ouro.nome_mae AS mae_cpf_ouro,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf,
    ouro.cep AS cep_cpf_ouro
'''

display(con.execute(f'''
SELECT
    av.unique_id_censo,
    av.unique_id_cpf_associado,
    av.unique_id_cpf_ouro,
    av.p_associado,
    po.match_probability AS p_ouro,
    CAST(
        ca.nome_completo IS NOT NULL AND pb.nome_completo IS NOT NULL
        AND ca.nome_completo = pb.nome_completo
        AS INTEGER
    ) AS nome_assoc,
    CAST(
        ca.nome_completo IS NOT NULL AND ouro.nome_completo IS NOT NULL
        AND ca.nome_completo = ouro.nome_completo
        AS INTEGER
    ) AS nome_ouro,
    CAST(
        ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
        AND ca.data_nascimento = pb.data_nascimento
        AS INTEGER
    ) AS dob_assoc,
    CAST(
        ca.data_nascimento IS NOT NULL AND ouro.data_nascimento IS NOT NULL
        AND ca.data_nascimento = ouro.data_nascimento
        AS INTEGER
    ) AS dob_ouro,
    CAST(
        ca.nome_mae IS NOT NULL AND pb.nome_mae IS NOT NULL
        AND ca.nome_mae = pb.nome_mae
        AS INTEGER
    ) AS mae_assoc,
    CAST(
        ca.nome_mae IS NOT NULL AND ouro.nome_mae IS NOT NULL
        AND ca.nome_mae = ouro.nome_mae
        AS INTEGER
    ) AS mae_ouro,
    {cols_exemplo}
FROM avaliacao_ouro av
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = av.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = av.unique_id_cpf_associado
JOIN {SPLINK_INPUT_VIEW} ouro ON ouro.unique_id = av.unique_id_cpf_ouro
LEFT JOIN splink_predictions po
    ON po.unique_id_censo = av.unique_id_censo
   AND po.unique_id_cpf = av.unique_id_cpf_ouro
WHERE av.corte = 'associou_cpf_errado'
ORDER BY av.p_associado DESC, av.unique_id_censo
LIMIT {TOP_N}
''').df())

display(con.execute(f'''
SELECT
    COUNT(*) AS n_associou_cpf_errado,
    COUNT(*) FILTER (
        WHERE ca.nome_completo IS NOT NULL AND pb.nome_completo IS NOT NULL
          AND ca.nome_completo = pb.nome_completo
          AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
          AND NOT (
              ouro.nome_completo IS NOT NULL
              AND ca.nome_completo = ouro.nome_completo
              AND ouro.data_nascimento IS NOT NULL
              AND ca.data_nascimento = ouro.data_nascimento
          )
    ) AS n_assoc_casa_ouro_nao,
    COUNT(*) FILTER (
        WHERE ca.nome_completo IS NOT NULL AND pb.nome_completo IS NOT NULL
          AND ca.nome_completo = pb.nome_completo
          AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
          AND ouro.nome_completo IS NOT NULL
          AND ca.nome_completo = ouro.nome_completo
          AND ouro.data_nascimento IS NOT NULL
          AND ca.data_nascimento = ouro.data_nascimento
    ) AS n_ambos_casam,
    COUNT(*) FILTER (
        WHERE po.match_probability IS NOT NULL
          AND po.match_probability >= av.p_associado
    ) AS n_p_ouro_ge_p_assoc
FROM avaliacao_ouro av
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = av.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = av.unique_id_cpf_associado
JOIN {SPLINK_INPUT_VIEW} ouro ON ouro.unique_id = av.unique_id_cpf_ouro
LEFT JOIN splink_predictions po
    ON po.unique_id_censo = av.unique_id_censo
   AND po.unique_id_cpf = av.unique_id_cpf_ouro
WHERE av.corte = 'associou_cpf_errado'
''').df())


,unique_id_censo,unique_id_cpf_associado,unique_id_cpf_ouro,p_associado,p_ouro,nome_assoc,nome_ouro,dob_assoc,dob_ouro,mae_assoc,...,nome_cpf_ouro,dob_censo,dob_cpf,dob_cpf_ouro,mae_censo,mae_cpf,mae_cpf_ouro,cep_censo,cep_cpf,cep_cpf_ouro
0,censo_2110856050000160110010000138000001000000007,cpf_06041422347,cpf_61090744382,1.0,1.000000,1,1,1,1,1,...,ISADORA KAROLINE SOUSA MELO,2002-03-09,2002-03-09,2002-03-09,CLAUDIANA SOUSA MELO,CLAUDIANA SOUSA MELO,CLAUDIANA SOUSA MELO,65929000,65929000,65929000
1,censo_2105427050000090090020000276000001000000003,cpf_03733565363,cpf_61053697350,1.0,1.000000,0,1,1,1,1,...,ADRIELY RODRIGUES REIS,1996-05-17,1996-05-17,1996-05-17,MARIA ROSEDINA SETUBAL REIS,MARIA ROSEDINA SETUBAL REIS,MARIA ROSEDINA SETUBAL REIS,65939000,65939000,65939000
2,censo_2105302050004280070010000185000001000000002,cpf_34325018387,cpf_43618677391,1.0,1.000000,1,1,1,1,1,...,IVALDO VIEIRA SOUSA,1968-08-11,1968-08-11,1968-08-11,ZELIA VIEIRA SOUSA,ZELIA VIEIRA SOUSA,ZELIA VIEIRA SOUSA,65903290,65903290,65903290
3,censo_2108058050000380210020000104000001000000003,cpf_61186614307,cpf_05581318328,1.0,NaN,1,1,1,0,1,...,ANTONIO JOSE SOUSA CONCEICAO,1995-08-20,1995-08-20,1961-12-25,EULILIA MARIA SOUSA CONCEICAO,EULILIA MARIA SOUSA CONCEICAO,ARACI SOUSA CONCEICAO,65585000,65585000,65585000
4,censo_2101608050002640010010000019000001000000002,cpf_04133128362,cpf_04220627316,1.0,1.000000,0,1,1,1,1,...,WISLLANDYA QUIRINO ROCHA,2001-03-20,2001-03-20,2001-03-20,ILRENIR SILVA QUIRINO,ILRENIR SILVA QUIRINO,ILRENIR SILVA QUIRONO ROCHA,65950000,65950000,65950000
5,censo_2101509050000260270010000197000001000000003,cpf_08427594330,cpf_61779168306,1.0,1.000000,1,0,1,1,1,...,DAMIAO CARVALHO SILVA,1998-04-11,1998-04-11,1998-04-11,MARIA CARMO CARVALHO,MARIA CARMO CARVALHO,MARIA CARMO CARVALHO,65660000,65660000,65660000
6,censo_2108900050000570050050000089000001000000003,cpf_61900906309,cpf_63404199332,1.0,1.000000,1,0,1,1,1,...,GLAUCIA OLIVEIRA SOUZA,2003-11-01,2003-11-01,2003-11-01,EDINEIA PEREIRA SOUSA,EDINEIA PEREIRA SOUSA,EDINEIA PEREIRA SOUZA,65740000,65740000,65740000
7,censo_2101905050000240000190000474000001000000003,cpf_11133202306,cpf_61801955360,1.0,1.000000,1,1,1,1,1,...,LUDMILA PEREIRA FRANCA,2005-07-20,2005-07-20,2005-07-20,PEDRA CARREIRA PEREIRA,PEDRA CARREIRA PEREIRA,PEDRA CARREIRA PEREIRA,65248000,65248000,65248000
8,censo_2100907050000800170040000223000001000000003,cpf_04901349392,cpf_60504417312,1.0,1.000000,0,1,1,1,1,...,DANIEL SOUZA PESSOA,1991-12-21,1991-12-21,1991-12-21,MARIA DORES MONTEIRO SOUZA,MARIA DORES MONTEIRO SOUZA,MARIA DORES MONTEIRO SOUZA,65570000,65570000,65570000
9,censo_2101202050000400110040000195000001000000002,cpf_87892359320,cpf_92720765368,1.0,1.000000,1,1,1,1,1,...,FRANCISCO ROCHA MACHADO,1981-12-03,1981-12-03,1981-12-03,DEUSIMAR ROCHA MACHADO,DEUSIMAR ROCHA MACHADO,DEUSIMAR ROCHA MACHADO,65700000,65700000,65700000


,n_associou_cpf_errado,n_assoc_casa_ouro_nao,n_ambos_casam,n_p_ouro_ge_p_assoc
0,3825,275,1560,2182


Amostra de FN por corte (tudo que não associou o CPF ouro).


In [23]:
display(con.execute(f'''
SELECT
    av.corte,
    av.unique_id_censo,
    av.unique_id_cpf_ouro,
    av.unique_id_cpf_associado,
    av.unique_id_cpf_escolha,
    av.p_associado,
    av.p_escolha,
    ca.nome_completo AS nome_censo,
    ouro.nome_completo AS nome_cpf_ouro,
    ca.data_nascimento AS dob_censo,
    ouro.data_nascimento AS dob_cpf_ouro
FROM avaliacao_ouro av
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = av.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ouro ON ouro.unique_id = av.unique_id_cpf_ouro
WHERE av.corte <> 'associou_cpf_ouro'
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY av.corte
    ORDER BY av.unique_id_censo
) <= {TOP_N}
ORDER BY av.corte, av.unique_id_censo
''').df())


,corte,unique_id_censo,unique_id_cpf_ouro,unique_id_cpf_associado,unique_id_cpf_escolha,p_associado,p_escolha,nome_censo,nome_cpf_ouro,dob_censo,dob_cpf_ouro
0,associou_cpf_errado,censo_2100055050000030070010000212000001000000001,cpf_66604710300,cpf_40283852372,cpf_40283852372,0.996327,0.996327,MARIA ALZENIRA VILAR,MARIA ALZENIRA WILAR SANTOS,1958-11-12,1958-11-12
1,associou_cpf_errado,censo_2100055050000060060080000330000001000000001,cpf_76731197300,cpf_34532650330,cpf_34532650330,0.999994,0.999994,ANTONIO MARCOS MORREIRA SANTOS,ANTONIO MARCOS MOREIRA SANTOS,1973-02-02,1973-02-02
2,associou_cpf_errado,censo_2100055050000090030010000040000001000000001,cpf_68154496372,cpf_30365694304,cpf_30365694304,1.000000,1.000000,GESSE COSTA PRIMO,GESSE COSTA PRIMO,1963-10-31,1963-10-31
3,associou_cpf_errado,censo_2100055050000140030030000199000001000000001,cpf_01385279206,cpf_85712957515,cpf_85712957515,0.998724,0.998724,MARIA DANTAS SANTOS,ELEN DAIANE BORGES DANTAS,1989-08-29,1989-08-29
4,associou_cpf_errado,censo_2100055050000260070020000194000001000000001,cpf_70327157372,cpf_69075352387,cpf_69075352387,1.000000,1.000000,FRANCISCA CHAGAS OLIVEIRA LIMA,FRANCISCA CHAGAS OLIVEIRA LIMA,1953-07-16,1953-07-16
5,associou_cpf_errado,censo_2100055050000260080040000376000001000000001,cpf_90008693315,cpf_00159029350,cpf_00159029350,0.988489,0.988489,RAQUEL LEAL SILVA,RAQUEL LEAL SALES,1981-05-11,1981-05-11
6,associou_cpf_errado,censo_2100055050000340160010000200000001000000002,cpf_65353536304,cpf_00201396327,cpf_00201396327,0.626842,0.626842,MARIA ALVES SANTOS,MARIA ALVES SILVA,1966-12-31,1966-12-31
7,associou_cpf_errado,censo_2100055050000380010020000001000001000000001,cpf_40282260315,cpf_36458120320,cpf_36458120320,0.999997,0.999997,IVANEIDE ALMEIDA NASCIMENTO,IVANEIDE ALMEIDA NASCIMENTO,1966-05-01,1966-05-01
8,associou_cpf_errado,censo_2100055050000380020020000011000001000000001,cpf_19771088300,cpf_60809356350,cpf_60809356350,0.973483,0.973483,FRANCISCO ASSIS PEREIRA,FRANCISCO ASSIS PEREIRA,1951-10-04,1951-10-04
9,associou_cpf_errado,censo_2100055050000390110030000294000002000000001,cpf_79647537387,cpf_77583388368,cpf_77583388368,0.999997,0.999997,WILTON CESAR OLIVEIRA LEMOS,WILTON CESAR OLIVEIRA LEMOS,1976-06-14,1976-06-14


## Waterfall de um par

Cola os `unique_id` (os de `splink_input`). Reroda só esta célula. Linker no
conjunto ouro × `cpf_limpo` — o TF é o desse recorte. Blocking só o par.


In [ ]:
from splink import Linker

UNIQUE_ID_CENSO = 'censo_COLOQUE_O_ID'
UNIQUE_ID_CPF = 'cpf_COLOQUE_O_ID'

n_censo_id = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}
WHERE unique_id = '{UNIQUE_ID_CENSO}' AND origem = 'censo'
''').fetchone()[0]
n_cpf_id = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}
WHERE unique_id = '{UNIQUE_ID_CPF}' AND origem = 'cpf'
''').fetchone()[0]
if n_censo_id != 1 or n_cpf_id != 1:
    raise RuntimeError(
        f'Ids não encontrados em splink_input: '
        f'censo {UNIQUE_ID_CENSO!r} n={n_censo_id}; '
        f'cpf {UNIQUE_ID_CPF!r} n={n_cpf_id}'
    )

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')

with open(SPLINK_MODEL_JSON, encoding='utf-8') as f:
    settings = json.load(f)
for comp in settings['comparisons']:
    if comp.get('output_column_name') != 'nome_completo_phon':
        continue
    for nivel in comp['comparison_levels']:
        if nivel.get('tf_adjustment_column') == 'primeiro_nome_phon':
            nivel['tf_adjustment_column'] = 'nome_completo_phon'
            nivel.pop('disable_tf_exact_match_detection', None)
settings['retain_intermediate_calculation_columns'] = True
settings['retain_matching_columns'] = True
settings['blocking_rules_to_generate_predictions'] = [
    {
        'blocking_rule': (
            f'l."unique_id" = \'{UNIQUE_ID_CENSO}\' '
            f'AND r."unique_id" = \'{UNIQUE_ID_CPF}\''
        ),
        'sql_dialect': 'duckdb',
    }
]

drop_splink_temp_tables(con)
db_api = get_splink_db_api(con)
censo_in = db_api.register('splink_censo')
censo_in.dataset_display_name = 'censo'
cpf_in = db_api.register('splink_cpf')
cpf_in.dataset_display_name = 'cpf'
linker_wf = Linker([censo_in, cpf_in], settings)
df_wf = linker_wf.inference.predict()
n_wf = con.execute(f'SELECT COUNT(*) FROM {df_wf.physical_name}').fetchone()[0]
if n_wf != 1:
    raise RuntimeError(f'Esperava 1 par no waterfall, veio {n_wf}')
row_wf = con.execute(f'''
SELECT match_probability, match_weight
FROM {df_wf.physical_name}
''').fetchone()
print('match_probability:', row_wf[0])
print('match_weight:', row_wf[1])

records = df_wf.as_record_dict(limit=1)
chart = linker_wf.visualisations.waterfall_chart(records)
drop_splink_temp_tables(con)
chart
